# Successive Halving with Optuna

Notebook for the course [Master Hyperparameter Optimization for Tabular Learning](http://www.trainindata.com/p/master-hyperparameter-optimization-for-tabular-learning)

In this notebook, we'll carry out [successive halving](https://optuna.readthedocs.io/en/stable/reference/generated/optuna.pruners.SuccessiveHalvingPruner.html) with Optuna.

In [1]:
import optuna

from sklearn.datasets import load_breast_cancer
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

import xgboost as xgb

# XGBoostPruningCallback moved out of optuna itself and into the
# separate optuna-integration package (pip install optuna-integration)
from optuna_integration import XGBoostPruningCallback

In [2]:
# load dataset and prepare data

data, target = load_breast_cancer(return_X_y=True)

X_train, X_test, y_train, y_test = train_test_split(
    data, target, test_size=0.25)

dtrain = xgb.DMatrix(X_train, label=y_train)
dtest = xgb.DMatrix(X_test, label=y_test)

## Define the objective function

Check out [xgboost pruning integration](https://optuna-integration.readthedocs.io/en/stable/reference/generated/optuna_integration.XGBoostPruningCallback.html#optuna_integration.XGBoostPruningCallback)

Code below based on https://github.com/optuna/optuna-examples/blob/main/xgboost/xgboost_integration.py

<div style="padding: 12px 16px; border-left: 5px solid #2196f3; background-color: #eaf4fd; border-radius: 4px;">
<strong>Note:</strong> For simplicity, this example uses <code>dtest</code> as validation data for pruning. In a real workflow, create a separate validation set for pruning and hyperparameter selection, and use the test set only once for the final evaluation.
</div>

In [3]:
def objective(trial):

    # hyperparameter space
    param = {
        "verbosity": 0,
        "objective": "binary:logistic",
        "eval_metric": "auc",
        "booster": trial.suggest_categorical("booster", ["gbtree", "gblinear", "dart"]),
        "lambda": trial.suggest_float("lambda", 1e-8, 1.0, log=True),
        "alpha": trial.suggest_float("alpha", 1e-8, 1.0, log=True),
    }

    # conditional space: some hyperparams depend on other hyperparams
    if param["booster"] == "gbtree" or param["booster"] == "dart":
        param["max_depth"] = trial.suggest_int("max_depth", 1, 9)
        param["eta"] = trial.suggest_float("eta", 1e-8, 1.0, log=True)
        param["gamma"] = trial.suggest_float("gamma", 1e-8, 1.0, log=True)
        param["grow_policy"] = trial.suggest_categorical("grow_policy", ["depthwise", "lossguide"])
    
    if param["booster"] == "dart":
        param["sample_type"] = trial.suggest_categorical("sample_type", ["uniform", "weighted"])
        param["normalize_type"] = trial.suggest_categorical("normalize_type", ["tree", "forest"])
        param["rate_drop"] = trial.suggest_float("rate_drop", 1e-8, 1.0, log=True)
        param["skip_drop"] = trial.suggest_float("skip_drop", 1e-8, 1.0, log=True)

    # Add a callback for pruning.
    # This is the stopping criterion for successive halving: ROC AUC after each round.
    pruning_callback = XGBoostPruningCallback(trial, "validation-auc")
    
    # set up the model
    bst = xgb.train(param, dtrain, evals=[(dtest, "validation")], callbacks=[pruning_callback])
    
    # evaluate
    preds = bst.predict(dtest)
    roc_auc = roc_auc_score(y_test, preds)
    
    return roc_auc

In the following code, we'll train **30 initial configurations**. The initial 30 configurations are sampled at random from the hyperparameter space.

As opposed to scikit-learn, Optuna does not find winning configurations and pass them to the next round. The successfull candidates are trained with more resources, while the unpromising ones are stopped earlier.

Like this, it trains less models.

In [4]:
study = optuna.create_study(
    
    # a way to sample hyperparameters to create the configurations
    sampler=optuna.samplers.RandomSampler(),
    
    # successive halving
    pruner=optuna.pruners.SuccessiveHalvingPruner(
        # controls the minimum validation rounds that it needs to wait before stopping
        min_resource=1,
        
        reduction_factor=3,
        
        # Minimum number of trials that need to complete a rung before any trial
        # is considered for promotion
        bootstrap_count = 0,
    ),
    
    direction="maximize",    
)


study.optimize(
    objective, 
    
    # the number of initial configurations
    n_trials=30, 
)

[I 2026-08-26 13:56:10,882] A new study created in memory with name: no-name-3d569a5f-9cba-444b-a191-f883794bcbfb


[0]	validation-auc:0.98344
[1]	validation-auc:0.98008
[2]	validation-auc:0.98113
[3]	validation-auc:0.98323
[4]	validation-auc:0.98574
[5]	validation-auc:0.98595
[6]	validation-auc:0.98616
[7]	validation-auc:0.98637
[8]	validation-auc:0.98721
[9]	validation-auc:0.98805


[I 2026-08-26 13:56:10,891] Trial 0 finished with value: 0.9880503144654088 and parameters: {'booster': 'gblinear', 'lambda': 1.380057025417775e-07, 'alpha': 2.4431555409383484e-05}. Best is trial 0 with value: 0.9880503144654088.


[0]	validation-auc:0.98795
[1]	validation-auc:0.98847
[2]	validation-auc:0.98847
[3]	validation-auc:0.98847
[4]	validation-auc:0.98920
[5]	validation-auc:0.98784
[6]	validation-auc:0.98784
[7]	validation-auc:0.98753
[8]	validation-auc:0.98753


[I 2026-08-26 13:56:10,907] Trial 1 pruned. Trial was pruned at iteration 9.


...

[I 2026-08-26 13:56:11,071] Trial 27 pruned. Trial was pruned at iteration 1.


[0]	validation-auc:0.98795


[I 2026-08-26 13:56:11,075] Trial 28 pruned. Trial was pruned at iteration 1.


[0]	validation-auc:0.97463


[I 2026-08-26 13:56:11,076] Trial 29 pruned. Trial was pruned at iteration 1.


In this particular case, `min_resource` controls the callback, that is, the minimum number of iterations / validations that it needs to wait before stopping the training.

If `min_resource=4` no model will be stopped until they undergo 4 rounds of validation. 

If `min_resource="auto"` then 1 is the minimum possible. A model can be stopped if in the first round it produces a score below previous models.

Change the `min_resource` and check it out.

In [5]:
# the best hyperparameters

study.best_params

{'booster': 'dart',
 'lambda': 1.0678531200340947e-05,
 'alpha': 0.021213959024626132,
 'max_depth': 5,
 'eta': 0.020433138008171977,
 'gamma': 0.06401877087204218,
 'grow_policy': 'lossguide',
 'sample_type': 'uniform',
 'normalize_type': 'tree',
 'rate_drop': 0.5256864797841302,
 'skip_drop': 0.0020167224184854606}

In [6]:
# the best performance value

study.best_value

0.989412997903564

In [7]:
r = study.trials_dataframe()

r

,number,value,datetime_start,datetime_complete,duration,params_alpha,params_booster,params_eta,params_gamma,params_grow_policy,params_lambda,params_max_depth,params_normalize_type,params_rate_drop,params_sample_type,params_skip_drop,system_attrs_completed_rung_0,system_attrs_completed_rung_1,system_attrs_completed_rung_2,state
0,0,0.988050,2026-08-26 13:56:10.882741,2026-08-26 13:56:10.891128,0 days 00:00:00.008387,2.443156e-05,gblinear,NaN,NaN,NaN,1.380057e-07,NaN,NaN,NaN,NaN,NaN,0.980084,0.983229,0.988050,COMPLETE
1,1,0.987945,2026-08-26 13:56:10.891349,2026-08-26 13:56:10.907908,0 days 00:00:00.016559,3.377987e-06,gbtree,2.742405e-02,2.401609e-05,depthwise,1.803904e-01,9.0,NaN,NaN,NaN,NaN,0.988470,0.988470,0.987945,PRUNED
2,2,0.958281,2026-08-26 13:56:10.908086,2026-08-26 13:56:10.909391,0 days 00:00:00.001305,8.466545e-01,gblinear,NaN,NaN,NaN,5.971957e-01,NaN,NaN,NaN,NaN,NaN,0.958281,NaN,NaN,PRUNED
3,3,0.858491,2026-08-26 13:56:10.909572,2026-08-26 13:56:10.911618,0 days 00:00:00.002046,1.017041e-05,dart,4.537032e-06,2.730140e-08,depthwise,3.428515e-07,1.0,tree,1.563179e-03,weighted,1.281197e-05,0.858491,NaN,NaN,PRUNED
4,4,0.987107,2026-08-26 13:56:10.911881,2026-08-26 13:56:10.916231,0 days 00:00:00.004350,9.964999e-01,dart,1.428992e-01,1.153777e-05,depthwise,5.813421e-05,6.0,tree,4.371894e-05,uniform,3.204619e-03,0.987107,NaN,NaN,PRUNED
5,5,0.966457,2026-08-26 13:56:10.916431,2026-08-26 13:56:10.917703,0 days 00:00:00.001272,1.707684e-02,gblinear,NaN,NaN,NaN,6.274840e-08,NaN,NaN,NaN,NaN,NaN,0.966457,NaN,NaN,PRUNED
6,6,0.971488,2026-08-26 13:56:10.917901,2026-08-26 13:56:10.923292,0 days 00:00:00.005391,1.339297e-05,dart,2.988499e-02,4.391257e-01,lossguide,2.598197e-04,7.0,forest,1.003209e-07,weighted,1.091141e-04,0.971488,NaN,NaN,PRUNED
7,7,0.971803,2026-08-26 13:56:10.923449,2026-08-26 13:56:10.928356,0 days 00:00:00.004907,4.282666e-08,gbtree,4.869213e-04,1.006761e-02,lossguide,1.029843e-05,6.0,NaN,NaN,NaN,NaN,0.971803,NaN,NaN,PRUNED
8,8,0.982600,2026-08-26 13:56:10.928508,2026-08-26 13:56:10.930265,0 days 00:00:00.001757,6.624920e-05,gblinear,NaN,NaN,NaN,5.956838e-06,NaN,NaN,NaN,NaN,NaN,0.982600,0.982600,NaN,PRUNED
9,9,0.988784,2026-08-26 13:56:10.930416,2026-08-26 13:56:10.952077,0 days 00:00:00.021661,8.791302e-07,gbtree,9.906501e-02,6.763611e-07,lossguide,1.492202e-08,4.0,NaN,NaN,NaN,NaN,0.989937,0.990042,0.988784,COMPLETE


In [8]:
# a "rung" is each round of successive halving

v = [v for v in r.columns if 'rung' in v]

30-r[v].isnull().sum()

system_attrs_completed_rung_0    30
system_attrs_completed_rung_1     9
system_attrs_completed_rung_2     6
dtype: int64

In [9]:
# some of the configurations from the last round of 
# successive halving, called "rung", were stopped early

r[~r["system_attrs_completed_rung_2"].isnull()]["state"]

0     COMPLETE
1       PRUNED
9     COMPLETE
18    COMPLETE
21      PRUNED
22    COMPLETE
Name: state, dtype: str

In [10]:
# completely trained configurations

r[r["state"]=="COMPLETE"]["state"].count()

np.int64(4)

As expected, we started with 30 configurations, roughly a third passed to the second round, and roughly a third passed to the third round.

It is not exactly a third, because this is asynchronous successive halving (ASHA), and we saw that in ASHA, some suboptimal configurations would be promoted to next rounds, because we don't wait to having them all to examine the top 30%.